In [1]:
import gc
import json
import os

import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import f1_score
from torch import GradScaler
from torch.cuda.amp import autocast
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from internal.data.mil_dataset import MILDatasetMemmapRanges
from internal.data.shuffle_and_cap_bag import ShuffleAndCapBag
from internal.nn.attention_mil import AttentionMIL

cuda_is_available = True
device = torch.device("cuda")

config = json.load(
    open(os.path.join("processed", "config.json"), "r")
)
OUT_DIR = config["OUT_DIR"]
X_PATH = config["X_PATH"]
M_PATH = config["M_PATH"]
Y_PATH = config["Y_PATH"]
IDX_PATH = config["IDX_PATH"]
LOG_PATH = config["LOG_PATH"]
TEST_X_PATH = config["TEST_X_PATH"]
TEST_M_PATH = config["TEST_M_PATH"]
TEST_IDX_PATH = config["TEST_IDX_PATH"]
TEST_LOG_PATH = config["TEST_LOG_PATH"]
PATCH_SIZE = config["PATCH_SIZE"]
N_PATCHES = config["N_PATCHES"]
MARGIN = config["MARGIN"]
MASK_PATCH_FRAC = config["MASK_PATCH_FRAC"]
MIN_MASK_PIXELS_SLIDE = config["MIN_MASK_PIXELS_SLIDE"]
MIN_MASK_IN_PATCH = config["MIN_MASK_IN_PATCH"]
MIN_TISSUE_FRAC = config["MIN_TISSUE_FRAC"]
MIN_PATCH_PER_SLIDE = config["MIN_PATCH_PER_SLIDE"]
MIN_CENTER_DIST = config["MIN_CENTER_DIST"]
MAX_TRIES_PER_SLIDE = config["MAX_TRIES_PER_SLIDE"]
MAX_TRIES_PER_PATCH = config["MAX_TRIES_PER_PATCH"]
CLASS2ID = config["CLASS2ID"]
CLASS2ID = {k: int(v) for k, v in CLASS2ID.items()}
ID2CLASS = config["ID2CLASS"]
ID2CLASS = {int(k): v for k, v in ID2CLASS.items()}

RETURN_META = True

In [2]:
def run_epoch(model, loader, optimizer=None, scaler=None, device="cuda"):
    train = optimizer is not None
    model.train(train)

    all_pred, all_true = [], []
    total_loss, n = 0.0, 0
    loss_fn = nn.CrossEntropyLoss(label_smoothing=0.05)

    # choose grad context
    grad_ctx = torch.enable_grad if train else torch.inference_mode

    for batch in tqdm(loader, desc="Train" if train else "Val", unit="batch"):
        xcat, y, bag_sizes = batch[:3]
        xcat = xcat.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        bag_sizes = bag_sizes.to(device, non_blocking=True)

        if train:
            optimizer.zero_grad(set_to_none=True)

        lambda_ent = 0.01
        with grad_ctx():
            with autocast(enabled=(scaler is not None) and train):
                logits, attn_ent = model(xcat, bag_sizes, return_attn_reg=True)
                loss = loss_fn(logits, y) - lambda_ent * attn_ent

            if train:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        total_loss += float(loss.detach().cpu())
        n += 1

        pred = logits.argmax(dim=1).detach().cpu().tolist()
        all_pred.extend(pred)
        all_true.extend(y.detach().cpu().tolist())

    macro_f1 = f1_score(all_true, all_pred, average="macro")
    return total_loss / max(n, 1), macro_f1

In [3]:
def mil_collate(batch):
    # batch items can be (x,y) or (x,y,meta)
    if len(batch[0]) == 3:
        xs, ys, metas = zip(*batch)
    else:
        xs, ys = zip(*batch)
        metas = None

    ys = torch.tensor(ys, dtype=torch.long)
    bag_sizes = torch.tensor([x.size(0) for x in xs], dtype=torch.long)
    xcat = torch.cat(xs, dim=0)  # (sum n_i, C, H, W)

    if metas is None:
        return xcat, ys, bag_sizes
    return xcat, ys, bag_sizes, metas

def mil_collate_test(batch):
    xs, metas = zip(*batch)
    bag_sizes = torch.tensor([x.size(0) for x in xs])
    xcat = torch.cat(xs, dim=0)
    return xcat, bag_sizes, metas

def mil_collate_with_meta(batch):
    xs = [b[0] for b in batch]
    ys = torch.stack([b[1] if torch.is_tensor(b[1]) else torch.tensor(b[1], dtype=torch.long) for b in batch]).long()
    metas = [b[2] for b in batch]  # list of dicts

    bag_sizes = torch.tensor([x.size(0) for x in xs], dtype=torch.long)
    xcat = torch.cat(xs, dim=0)

    return xcat, ys, bag_sizes, metas

def mil_collate_concat(batch):
    if RETURN_META:
        xs, ys, _ = zip(*batch)   # each x: (n_i,C,H,W)
    else:
        xs, ys = zip(*batch)      # each x: (n_i,C,H,W)
    bag_sizes = torch.tensor([x.shape[0] for x in xs], dtype=torch.long)
    x = torch.cat(xs, dim=0)  # (sum n_i, C,H,W)
    y = torch.stack(ys)       # (B,)
    return x, y, bag_sizes

In [4]:
from internal.data.shuffle_and_random_cap_bag import ShuffleAndRandomCapBag

train_ds = MILDatasetMemmapRanges(
    x_path=X_PATH,
    y_path=Y_PATH,
    idx_path=IDX_PATH,
    m_path=M_PATH,
    patch_size=PATCH_SIZE,
    bag_transform=ShuffleAndRandomCapBag(),
    return_meta=True
)
val_ds   = MILDatasetMemmapRanges(
    x_path=X_PATH,
    y_path=Y_PATH,
    idx_path=IDX_PATH,
    m_path=M_PATH,
    patch_size=PATCH_SIZE,
    bag_transform=ShuffleAndCapBag(max_instances=16),
    return_meta=True
)

train_loader = DataLoader(
    train_ds,
    batch_size=2,              # number of slides per batch
    shuffle=True,
    # num_workers=0,
    pin_memory=cuda_is_available,
    collate_fn=mil_collate_concat,   # or mil_collate_list
)
val_loader = DataLoader(
    val_ds,
    batch_size=1,
    shuffle=False,
    # num_workers=0,
    pin_memory=False,
    collate_fn=mil_collate,
)

In [5]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

model = AttentionMIL(n_classes=4).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=1e-3)
scaler = GradScaler(enabled=(device=="cuda"))

best_f1 = -1
best_epoch = 0
patience = 5
for epoch in range(1, 21):
    print(f"Epoch {epoch:02d} -------------------------------")
    tr_loss, tr_f1 = run_epoch(model, train_loader, optimizer=optimizer, scaler=scaler, device=device)
    gc.collect()
    if cuda_is_available:
        torch.cuda.empty_cache()
    va_loss, va_f1 = run_epoch(model, val_loader, optimizer=None, scaler=None, device=device)

    print(f"Epoch {epoch:02d} | train loss {tr_loss:.4f} f1 {tr_f1:.4f} | val loss {va_loss:.4f} f1 {va_f1:.4f}")

    if va_f1 > best_f1:
        print(f"New best val F1: {va_f1:.4f} (prev {best_f1:.4f}), saving model.")
        best_f1 = va_f1
        best_epoch = epoch
        torch.save({"model": model.state_dict()}, os.path.join(OUT_DIR, f"best_mil_f1_{best_f1:4f}.pt"))
    elif epoch - best_epoch >= patience:
        print(f"Early stop (best epoch={best_epoch}, best val f1={best_f1:.4f})")
        break


Epoch 01 -------------------------------


Train:   0%|          | 0/291 [00:00<?, ?batch/s]

Val:   0%|          | 0/581 [00:00<?, ?batch/s]

Epoch 01 | train loss 1.3227 f1 0.1644 | val loss 1.3230 f1 0.1780
New best val F1: 0.1780 (prev -1.0000), saving model.
Epoch 02 -------------------------------


Train:   0%|          | 0/291 [00:00<?, ?batch/s]

...

Train:   0%|          | 0/291 [00:00<?, ?batch/s]

Val:   0%|          | 0/581 [00:00<?, ?batch/s]

Epoch 07 | train loss 0.7947 f1 0.6590 | val loss 1.6871 f1 0.2572
Epoch 08 -------------------------------


Train:   0%|          | 0/291 [00:00<?, ?batch/s]

KeyboardInterrupt: 

In [ ]:
class CapBag:
    def __init__(self, max_instances=None):
        self.max_instances = max_instances
    def __call__(self, x):
        if self.max_instances is not None and x.size(0) > self.max_instances:
            return x[:self.max_instances]
        return x

test_ds = MILDatasetMemmapRanges(
    x_path=TEST_X_PATH,
    m_path=TEST_M_PATH,
    idx_path=TEST_IDX_PATH,
    # y_path=None,                    # no labels
    patch_size=PATCH_SIZE,
    bag_transform=CapBag(16),        # or None if fits
    return_meta=True
)

test_loader = DataLoader(
    test_ds,
    batch_size=1,          # always 1 for MIL inference
    shuffle=False,
    num_workers=os.cpu_count() // 2,         # avoids OS kill
    pin_memory=False,
    collate_fn=mil_collate_test,
)

model = AttentionMIL(n_classes=4, enc_chunk=2)  # same as training
checkpoint = torch.load("best_mil.pt", map_location="cpu")
model.load_state_dict(checkpoint["model"])
model.to(device)
model.eval()

id2class = ID2CLASS

pred_rows = []

with torch.inference_mode():
    for batch in tqdm(test_loader, desc="Inference", unit="slide"):
        xcat, bag_sizes, meta = batch  # y is missing
        xcat = xcat.to(device, non_blocking=True)  # keep float32
        bag_sizes = bag_sizes.to(device)

        with torch.inference_mode(), torch.cuda.amp.autocast(enabled=(device=="cuda")):
            logits = model(xcat, bag_sizes)

        pred = logits.argmax(dim=1).item()

        slide_id = meta[0]["slide_id"] if "slide_id" in meta[0] else meta[0]["slide_index"]
        pred_rows.append({
            "sample_index": f"img_{str(slide_id).zfill(4)}.png",
            "label": id2class[pred]
        })

sub_df = pd.DataFrame(pred_rows)
sub_df = sub_df.sort_values("sample_index")

sub_df.to_csv("submission.csv", index=False)
print("Saved submission.csv")

In [ ]:
sub_df["label"].value_counts(normalize=True)

In [ ]:
class CapBag:
    def __init__(self, max_instances: int = 16):
        self.max_instances = max_instances

    def __call__(self, x):
        if self.max_instances is None or x.size(0) <= self.max_instances:
            return x
        return x[:self.max_instances]   # deterministic

import torch.nn.functional as F

def apply_tta(x, tta_id: int):
    # x: (N, C, H, W)
    if tta_id == 0:
        return x
    if tta_id == 1:
        return torch.flip(x, dims=[3])        # Hflip (W)
    if tta_id == 2:
        return torch.flip(x, dims=[2])        # Vflip (H)
    if tta_id == 3:
        return torch.rot90(x, k=1, dims=[2,3])  # 90°
    if tta_id == 4:
        return torch.rot90(x, k=2, dims=[2,3])  # 180°
    if tta_id == 5:
        return torch.rot90(x, k=3, dims=[2,3])  # 270°
    if tta_id == 6:
        return torch.flip(torch.rot90(x, 1, [2,3]), dims=[3])  # rot90 + hflip
    if tta_id == 7:
        return torch.flip(torch.rot90(x, 1, [2,3]), dims=[2])  # rot90 + vflip
    raise ValueError("bad tta_id")

@torch.no_grad()
def predict_batch_tta(model, xcat, bag_sizes, device="cuda", n_tta=4, use_amp=True):
    model.eval()

    probs_acc = None
    for t in tqdm(range(n_tta), desc="TTA", unit="tta"):
        xt = apply_tta(xcat, t).to(device, non_blocking=True)
        bs = bag_sizes.to(device, non_blocking=True)

        with autocast(enabled=(use_amp and device == "cuda")):
            logits = model(xt, bs)              # (B, n_classes)
            probs  = F.softmax(logits, dim=1)   # (B, n_classes)

        probs_acc = probs if probs_acc is None else (probs_acc + probs)

    probs_mean = probs_acc / float(n_tta)
    pred = probs_mean.argmax(dim=1)
    return pred, probs_mean

test_ds = MILDatasetMemmapRanges(
    x_path=TEST_X_PATH,
    idx_path=TEST_IDX_PATH,
    m_path=TEST_M_PATH,
    patch_size=PATCH_SIZE,
    bag_transform=CapBag(max_instances=16),
    return_meta=True
)

test_loader = DataLoader(
    test_ds,
    batch_size=1,                   # keep 1 for low VRAM
    shuffle=False,
    num_workers=0,
    pin_memory=(device == "cuda"),
    collate_fn=mil_collate_test,
)

# ---- load model ----
model = AttentionMIL(n_classes=4)   # use your exact constructor
ckpt = torch.load("best_mil.pt", map_location="cpu")
model.load_state_dict(ckpt["model"] if isinstance(ckpt, dict) and "model" in ckpt else ckpt)
model.to(device)

# ---- inference + TTA ----
rows = []
N_TTA = 6   # start with 4, later try 6

for xcat, bag_sizes, metas in tqdm(test_loader, desc="Test TTA", unit="slide"):
    # metas is a tuple with len=1 because batch_size=1
    meta0 = metas[0]

    pred, probs = predict_batch_tta(model, xcat, bag_sizes, device=device, n_tta=N_TTA, use_amp=True)
    pred_id = int(pred.item())

    slide_id = meta0.get("slide_id", None)
    if slide_id is None:
        slide_id = meta0["slide_index"]

    rows.append({"sample_index": f"img_{str(slide_id).zfill(4)}.png", "label": pred_id})

sub = pd.DataFrame(rows)
sub.to_csv("submission.csv", index=False)
print("Saved submission.csv with", len(sub), "rows")

In [ ]:
sub_df["label"].value_counts(normalize=True)